# MLflow Example: Tracking Two Model Training Runs

In [1]:
# === Model Training and Experimentation ===

# Import necessary libraries
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_digits


In [2]:
# Load dataset
data = load_digits()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=42
)


# Define a function to train a model and return its accuracy
def train_model(n_estimators, max_depth):
    # Train a RandomForestClassifier
    model = RandomForestClassifier(
        n_estimators=n_estimators, max_depth=max_depth, random_state=42
    )
    model.fit(X_train, y_train)

    # Make predictions
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)

    print(
        f"Model with n_estimators={n_estimators}, max_depth={max_depth} achieved accuracy={accuracy:.4f}"
    )
    return model, accuracy

In [3]:
# Train models with different hyperparameter configurations
print("Training Model 1...")
model_1, acc_1 = train_model(n_estimators=20, max_depth=5)

Training Model 1...
Model with n_estimators=20, max_depth=5 achieved accuracy=0.9389


In [4]:
print("\nTraining Model 2...")
model_2, acc_2 = train_model(n_estimators=100, max_depth=10)



Training Model 2...
Model with n_estimators=100, max_depth=10 achieved accuracy=0.9722


Now adding MLFlow

In [5]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_digits

In [6]:
# get this code from dagshub
import dagshub

dagshub.init(repo_owner="ChenAuCarre", repo_name="mlflow-with-dagshub", mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

/Users/chenchen/Dev/mlflow-with-dagshub/.venv/lib/python3.12/site-packages/rich/live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=430e6e30-af18-486f-8809-f0638266d40a&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=dd9220b5f2372a521494424c60741aac905372605a2dac731249c77f467c4db0




Accessing as ChenAuCarre

Initialized MLflow to track repo "ChenAuCarre/mlflow-with-dagshub"

Repository ChenAuCarre/mlflow-with-dagshub initialized!

In [7]:
data = load_digits()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=42
)

In [8]:
# Set MLflow experiment name. This will get created if it doesn't exist
experiment_name = "RandomForest_V1.0.1"
mlflow.set_experiment(experiment_name)


<Experiment: artifact_location='mlflow-artifacts:/d76e89807ce34ea08e348d5b7b15647f', creation_time=1794844716342, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1794844716342, lifecycle_stage='active', name='RandomForest_V1.0.1', tags={}, trace_location=None, workspace='default'>

In [9]:
# Define a function to train a model, log parameters and metrics to MLflow
def train_and_log_model(n_estimators, max_depth, run_name):
    with mlflow.start_run(run_name=run_name):
        # Train a RandomForestClassifier
        model = RandomForestClassifier(
            n_estimators=n_estimators, max_depth=max_depth, random_state=42
        )
        model.fit(X_train, y_train)

        # Make predictions
        predictions = model.predict(X_test)
        accuracy = accuracy_score(y_test, predictions)

        # Log parameters, metrics, and the model itself
        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_metric("accuracy", accuracy)
        # MLflow 3 serializes with skops, which rejects tree types unless trusted
        mlflow.sklearn.log_model(
            model,
            name="model",
            registered_model_name="digits-rf",  # new version per run; API serves the latest
            skops_trusted_types=["sklearn.tree._tree.Tree"],
        )

        print(
            f"Logged RandomForest model with n_estimators={n_estimators}, max_depth={max_depth}, accuracy={accuracy:.4f}"
        )


In [10]:
# Train and log Model 1
print("Training and Logging Model 1...")
train_and_log_model(n_estimators=20, max_depth=5, run_name="random_forest_model_run_1")

Training and Logging Model 1...


2026/11/20 15:51:10 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Successfully registered model 'digits-rf'.
2026/11/20 15:51:16 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: digits-rf, version 1
Created version '1' of model 'digits-rf'.


Logged RandomForest model with n_estimators=20, max_depth=5, accuracy=0.9389
🏃 View run random_forest_model_run_1 at: https://dagshub.com/ChenAuCarre/mlflow-with-dagshub.mlflow/#/experiments/0/runs/c9c5c33027c04455a101cddbc4e419bc
🧪 View experiment at: https://dagshub.com/ChenAuCarre/mlflow-with-dagshub.mlflow/#/experiments/0


In [11]:
# Train and log Model 2
print("\nTraining and Logging Model 2...")
train_and_log_model(
    n_estimators=100, max_depth=10, run_name="random_forest_model_run_V1_0_2"
)


Training and Logging Model 2...


2026/11/20 15:51:24 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered model 'digits-rf' already exists. Creating a new version of this model...
2026/11/20 15:51:42 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: digits-rf, version 2
Created version '2' of model 'digits-rf'.


Logged RandomForest model with n_estimators=100, max_depth=10, accuracy=0.9722
🏃 View run random_forest_model_run_V1_0_2 at: https://dagshub.com/ChenAuCarre/mlflow-with-dagshub.mlflow/#/experiments/0/runs/6dd5b1f1924249ecbc2c910c69e971cf
🧪 View experiment at: https://dagshub.com/ChenAuCarre/mlflow-with-dagshub.mlflow/#/experiments/0


In [12]:
# Instructions to visualize results
# print("\nTo view the results, run the following command in your terminal:")
# print("mlflow ui")
# print("Then navigate to http://127.0.0.1:5000 to explore the experiment results.")

# Runs are now tracked on DagsHub, not a local server. View them here:
print("Experiment & runs: https://dagshub.com/ChenAuCarre/mlflow-with-dagshub.mlflow")
print("Or in the DagsHub repo: Experiments tab")

Experiment & runs: https://dagshub.com/ChenAuCarre/mlflow-with-dagshub.mlflow
Or in the DagsHub repo: Experiments tab
